In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [31]:
# File paths
BV = 'data/BV.csv'
PW = 'data/PW.csv'
SAM = 'data/SAM.csv'
PA = 'data/PA.csv'

# Load datasets
bv  = pd.read_csv(BV,  dtype={'sam_id': str})
pw  = pd.read_csv(PW,  dtype={'sam_id': str})
sam = pd.read_csv(SAM, dtype={'SAM_ADDRESS_ID': str, 'PARCEL': str}, low_memory=False)
pa  = pd.read_csv(PA,  dtype={'PID': str}, low_memory=False)
pa.columns = pa.columns.str.strip()

In [32]:
print(f'Building Violations before cleaning: {len(bv):,} rows')

# Drop rows with sam_id is missing
bv_clean = bv.dropna(subset=['sam_id'])

# Drop dups
bv_clean = bv_clean.drop_duplicates()

# Strip whitespace
bv_clean['sam_id'] = bv_clean['sam_id'].str.strip()

# Parse for violation date
date_col = [c for c in bv_clean.columns if 'date' in c.lower() or 'dttm' in c.lower()]

for col in date_col:
    bv_clean[col] = pd.to_datetime(bv_clean[col], errors='coerce')

# Standardize: strip whitespace, uppercase
str_cols = bv_clean.select_dtypes(include='object').columns
for col in str_cols:
    bv_clean[col] = bv_clean[col].str.strip()

print(f'\nBuilding Violations after cleaning: {len(bv_clean):,} rows')
bv_clean.head(3)

Building Violations before cleaning: 17,183 rows

Building Violations after cleaning: 17,075 rows


C:\Users\adria\AppData\Local\Temp\ipykernel_94284\1755956381.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = bv_clean.select_dtypes(include='object').columns


,case_no,ap_case_defn_key,status_dttm,status,code,value,description,violation_stno,violation_sthigh,violation_street,violation_suffix,violation_city,violation_state,violation_zip,ward,contact_addr1,contact_addr2,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
0,V91983,1013,NaT,Closed,121.2,NaN,Unsafe and Dangerous,302,NaN,Sumner,ST,East Boston,MA,02128,01,302 Sumner St,NaN,East Boston,MA,02128,132380,42.367679,-71.036580,"(42.36767899977675, -71.03658000178699)"
1,V898855,1013,2026-03-25 10:13:31,Open,105.1,NaN,Failure to Obtain Permit,335,347,Harrison,AV,Roxbury,MA,02118,03,1745 SHEA CENTER DR #200,NaN,HIGHLANDS RANCH,CO,80129,355636,42.345034,-71.064197,"(42.34503376751217, -71.06419739751462)"
2,V898828,1013,2026-03-25 09:20:39,Open,105.1,NaN,Failure to Obtain Permit,217,NaN,Albany,ST,Roxbury,MA,02118,03,2310 WASHINGTON ST,NaN,NEWTON LOWER FALLS,MA,02462,1533,42.345440,-71.061720,"(42.34544000003873, -71.06172000128106)"


In [33]:
print(f'SAM before cleaning: {len(sam):,} rows')

# Drop rows missing either join key
sam_clean = sam.dropna(subset=['SAM_ADDRESS_ID', 'PARCEL'])

# Drop dups
sam_clean = sam_clean.drop_duplicates(subset=['SAM_ADDRESS_ID'])

# Convert both join keys to string so they match
sam_clean['SAM_ADDRESS_ID'] = sam_clean['SAM_ADDRESS_ID'].astype(str).str.strip()
sam_clean['PARCEL'] = sam_clean['PARCEL'].astype(str).str.strip()

print(f'\nSAM after cleaning: {len(sam_clean):,} rows')
sam_clean.head(3)

SAM before cleaning: 399,661 rows

SAM after cleaning: 399,661 rows


,SAM_ADDRESS_ID,BUILDING_ID,RELATIONSHIP_TYPE,FULL_ADDRESS,STREET_NUMBER,IS_RANGE,RANGE_FROM,RANGE_TO,UNIT,FULL_STREET_NAME,STREET_ID,STREET_PREFIX,STREET_BODY,STREET_SUFFIX_ABBR,STREET_FULL_SUFFIX,STREET_SUFFIX_DIR,STREET_NUMBER_SORT,MAILING_NEIGHBORHOOD,ZIP_CODE,X_COORD,Y_COORD,SAM_STREET_ID,WARD,PARCEL,created_date,last_edited_date,shape_wkt,POINT_X,POINT_Y
0,1,100778,1,6-10 A St,6-10,1,6,10,NaN,A St,1,NaN,A,St,Street,NaN,6,Hyde Park,2136,757684.428458,2.916575e+06,1,18,1809309000,9/25/2009 10:14:59,10/25/2017 14:04:04,POINT (-71.125035941999954 42.250617902000045),-71.125036,42.250618
1,11,154909,1,10 A St,10,0,NaN,NaN,NaN,A St,2,NaN,A,St,Street,NaN,10,South Boston,2127,775987.559175,2.949557e+06,2,6,0600090000,9/25/2009 10:14:59,2/10/2022 10:47:25,POINT (-71.056799999999953 42.340880001000073),-71.056800,42.340880
2,17,141252,1,176-178 A St,176-178,1,176,178,NaN,A St,2,NaN,A,St,Street,NaN,176,Boston,2210,776990.886893,2.951048e+06,2,6,0601169000,9/25/2009 10:14:59,2/10/2022 10:47:31,POINT (-71.053059999999959 42.344958000000076),-71.053060,42.344958


In [35]:
print(f'Property Assessment before cleaning: {len(pa):,} rows')

# Drop rows missing PID
pa_clean = pa.dropna(subset=['PID'])

# Drop dups
pa_clean = pa_clean.drop_duplicates(subset=['PID'])

# Strip whitespace
pa_clean['PID'] = pa_clean['PID'].str.strip()

if 'OWNER' in pa_clean.columns:
    pa_clean['OWNER'] = pa_clean['OWNER'].str.strip().str.upper()

# Cast YR_BUILT to int
if 'YR_BUILT' in pa_clean.columns:
    pa_clean['YR_BUILT'] = pd.to_numeric(pa_clean['YR_BUILT'], errors='coerce')

# Cap YR_BUILT
pa_clean = pa_clean[(pa_clean['YR_BUILT'].isna()) | 
                    ((pa_clean['YR_BUILT'] >= 1600) & (pa_clean['YR_BUILT'] <= 2025))]
print(f'\nProperty Assessment after cleaning: {len(pa_clean):,} rows')
pa_clean.head(3)

Property Assessment before cleaning: 184,552 rows

Property Assessment after cleaning: 184,551 rows


,PID,CM_ID,GIS_ID,ST_NUM,ST_NUM2,ST_NAME,UNIT_NUM,CITY,ZIP_CODE,BLDG_SEQ,NUM_BLDGS,LUC,LU,LU_DESC,BLDG_TYPE,OWN_OCC,OWNER,MAIL_ADDRESSEE,MAIL_STREET_ADDRESS,MAIL_CITY,MAIL_STATE,MAIL_ZIP_CODE,RES_FLOOR,CD_FLOOR,RES_UNITS,...,EXT_FNISHED,INT_COND,EXT_COND,OVERALL_COND,BED_RMS,FULL_BTH,HLF_BTH,KITCHENS,TT_RMS,BDRM_COND,BTHRM_STYLE1,BTHRM_STYLE2,BTHRM_STYLE3,KITCHEN_TYPE,KITCHEN_STYLE1,KITCHEN_STYLE2,KITCHEN_STYLE3,HEAT_TYPE,HEAT_SYSTEM,AC_TYPE,FIREPLACES,ORIENTATION,NUM_PARKING,PROP_VIEW,CORNER_UNIT
0,0100001000,NaN,100001000,195.0,NaN,Lexington ST,NaN,EAST BOSTON,2128.0,1,1,105,R3,THREE-FAM DWELLING,RE - Row End,Y,PASCUCCI CARLO,NaN,195 LEXINGTON ST,EAST BOSTON,MA,02128,3.0,NaN,NaN,...,A - Asbestos,A - Average,F - Fair,A - Average,6.0,3.0,0.0,3.0,12.0,NaN,S - Semi-Modern,S - Semi-Modern,S - Semi-Modern,3F - 3 Full Eat In Kitchens,S - Semi-Modern,S - Semi-Modern,S - Semi-Modern,W - Ht Water/Steam,NaN,N - None,0.0,NaN,3.0,A - Average,NaN
1,0100002000,NaN,100002000,197.0,NaN,Lexington ST,NaN,EAST BOSTON,2128.0,1,1,105,R3,THREE-FAM DWELLING,RM - Row Middle,N,SEMBRANO LIVING TRUST,NaN,2 MITCHELL LANE,WAKEFIELD,MA,01880,3.0,NaN,NaN,...,M - Vinyl,A - Average,A - Average,A - Average,3.0,3.0,0.0,3.0,9.0,NaN,M - Modern,M - Modern,M - Modern,3F - 3 Full Eat In Kitchens,M - Modern,M - Modern,M - Modern,F - Forced Hot Air,NaN,C - Central AC,0.0,NaN,0.0,A - Average,NaN
2,0100003000,NaN,100003000,199.0,NaN,Lexington ST,NaN,EAST BOSTON,2128.0,1,1,105,R3,THREE-FAM DWELLING,RM - Row Middle,N,GUERRA CHEVARRIA ANA S,NaN,199 LEXINGTON ST,EAST BOSTON,MA,02128,3.0,NaN,NaN,...,M - Vinyl,A - Average,G - Good,A - Average,5.0,3.0,0.0,3.0,13.0,NaN,M - Modern,M - Modern,M - Modern,3F - 3 Full Eat In Kitchens,S - Semi-Modern,S - Semi-Modern,S - Semi-Modern,S - Space Heat,NaN,N - None,0.0,NaN,0.0,A - Average,NaN


In [36]:
# Merge Building Violations + SAM Addresses
merged_1 = bv_clean.merge(
    sam_clean,
    left_on='sam_id',
    right_on='SAM_ADDRESS_ID',
    how='left',
    suffixes=('_bv', '_sam')
)

# Merge Result + Property Assessment
merged_2 = merged_1.merge(
    pa_clean,
    left_on='PARCEL',
    right_on='PID',
    how='left',
    suffixes=('', '_pa')
)

matched_2    = merged_2['PID'].notna().sum()
unmatched_2  = merged_2['PID'].isna().sum()
match_rate_2 = matched_2 / len(merged_2) * 100

print(f'\nAfter Merge:')
print(f'Total rows: {len(merged_2):,}')
print(f'Matched: {matched_2:,} ({match_rate_2:.1f}%)')
print(f'Unmatched: {unmatched_2:,} ({100 - match_rate_2:.1f}%)')


After Merge:
Total rows: 17,075
Matched: 16,271 (95.3%)
Unmatched: 804 (4.7%)


In [37]:
print('All columns in merged dataset:')
for col in merged_2.columns:
    print(' ', col)

All columns in merged dataset:
  case_no
  ap_case_defn_key
  status_dttm
  status
  code
  value
  description
  violation_stno
  violation_sthigh
  violation_street
  violation_suffix
  violation_city
  violation_state
  violation_zip
  ward
  contact_addr1
  contact_addr2
  contact_city
  contact_state
  contact_zip
  sam_id
  latitude
  longitude
  location
  SAM_ADDRESS_ID
  BUILDING_ID
  RELATIONSHIP_TYPE
  FULL_ADDRESS
  STREET_NUMBER
  IS_RANGE
  RANGE_FROM
  RANGE_TO
  UNIT
  FULL_STREET_NAME
  STREET_ID
  STREET_PREFIX
  STREET_BODY
  STREET_SUFFIX_ABBR
  STREET_FULL_SUFFIX
  STREET_SUFFIX_DIR
  STREET_NUMBER_SORT
  MAILING_NEIGHBORHOOD
  ZIP_CODE
  X_COORD
  Y_COORD
  SAM_STREET_ID
  WARD
  PARCEL
  created_date
  last_edited_date
  shape_wkt
  POINT_X
  POINT_Y
  PID
  CM_ID
  GIS_ID
  ST_NUM
  ST_NUM2
  ST_NAME
  UNIT_NUM
  CITY
  ZIP_CODE_pa
  BLDG_SEQ
  NUM_BLDGS
  LUC
  LU
  LU_DESC
  BLDG_TYPE
  OWN_OCC
  OWNER
  MAIL_ADDRESSEE
  MAIL_STREET_ADDRESS
  MAIL_CITY
  MAIL_

In [38]:
keep_cols = [
    # From BV — violation info 
    'case_no',              # unique violation identifier
    'status_dttm',          # date of violation
    'status',               # open/closed
    'code',                 # violation code
    'description',          # violation type
    'violation_stno',       # street number
    'violation_street',     # street name
    'violation_city',       # neighborhood
    'violation_zip',        # zip code
    'ward',                 # city ward
    'sam_id',               # join key
    'latitude',             # for mapping
    'longitude',

    # From SAM — location info
    'MAILING_NEIGHBORHOOD', # neighborhood
    'PARCEL',               # join key to PA
    'POINT_X',              # coordinates for mapping
    'POINT_Y',

    # From PA — property/owner info
    'OWNER',                # landlord/management company
    'YR_BUILT',             # build year
    'LU',                   # land use code
    'LU_DESC',              # land use description
    'BLDG_TYPE',            # building type
    'OWN_OCC',              # owner occupied Y/N
    'OVERALL_COND',         # building condition
]

df_final = merged_2[keep_cols]
print(f'Final dataset: {df_final.shape[0]:,} rows, {df_final.shape[1]} columns')
df_final.head(3)

Final dataset: 17,075 rows, 24 columns


,case_no,status_dttm,status,code,description,violation_stno,violation_street,violation_city,violation_zip,ward,sam_id,latitude,longitude,MAILING_NEIGHBORHOOD,PARCEL,POINT_X,POINT_Y,OWNER,YR_BUILT,LU,LU_DESC,BLDG_TYPE,OWN_OCC,OVERALL_COND
0,V91983,NaT,Closed,121.2,Unsafe and Dangerous,302,Sumner,East Boston,02128,01,132380,42.367679,-71.036580,East Boston,0104910000,-71.036580,42.367679,ORELLANA-SERRANO ISRAEL,1930.0,R3,THREE-FAM DWELLING,RM - Row Middle,Y,A - Average
1,V898855,2026-03-25 10:13:31,Open,105.1,Failure to Obtain Permit,335,Harrison,Roxbury,02118,03,355636,42.345034,-71.064197,Roxbury,0306337000,-71.064197,42.345034,345 HARRISON LLC,2016.0,RC,RES /COMMERCIAL USE,120 - LUXURY APARTMENT,N,EX - Excellent
2,V898828,2026-03-25 09:20:39,Open,105.1,Failure to Obtain Permit,217,Albany,Roxbury,02118,03,1533,42.345440,-71.061720,Roxbury,0306626000,-71.061720,42.345440,217 ALBANY II LLC,2020.0,A,LUXURY APARTMENT,114 - APT 100+ UNITS,N,VG - Very Good


In [21]:
df_final.to_csv('merged_violations.csv', index=False)
print('Saved merged_violations_clean.csv')
print(f'Shape: {df_final.shape}')

Saved merged_violations_clean.csv
Shape: (17075, 24)
